# NeuroCode: Budget-Constrained Heterogeneous Code-Graph Slicing for Local Software Repair
### Google DeepMind - The Gemma 4 Developer Agent Paper Track (14B Model SWE-bench Lite Evaluation)

**Author:** Deepak R. (Department of Robotics & Automation Engineering, Dhaanish Ahmed College of Engineering, Anna University)  
**Email:** `wssedd18@gmail.com` | **Portfolio:** [https://deepak-arkz.me](https://deepak-arkz.me)  
**GitHub Repository:** [https://github.com/Arkz-Deepak/neurocode](https://github.com/Arkz-Deepak/neurocode)  

---

## Executive Summary
Autonomous software engineering (SWE) agents predominantly rely on proprietary cloud models with 128k+ token context windows running on 80GB datacenter GPUs. For developers on commodity consumer hardware (e.g., 4GB--8GB VRAM laptops or single 16GB accelerators), repository-scale agentic coding is blocked by three bottlenecks:
1. **The KV-Cache Memory Wall:** Ingesting 25,000+ token codebases demands upwards of 17.2 GB of VRAM for KV cache activations alone under 16-bit precision, instantly triggering CUDA Out-of-Memory (OOM) failures.
2. **Exploratory Tool-Loop Drift:** Small models choke during multi-turn ReAct loops (`grep`, `find`, `cat`), falling into repetitive syntax hallucinations.
3. **Lexical Blindness:** Flat keyword search (BM25) fails when bug reports describe symptoms at an API entrypoint while the root cause lies in an indirect callee lacking keyword overlap.

**NeuroCode** resolves this by establishing a clean division of labor:
* **Host CPU & RAM:** Rapidly constructs an in-memory Code Property Graph (CPG) via Tree-sitter AST parsing and computes a topological $k$-hop reachability closure packed via a formal Multiple-Choice Knapsack Problem (MCKP) algorithm.
* **Accelerator / Local Model:** Receives a bounded, high-density context slice ($\le 1,500$ tokens) for single-turn patch synthesis. On a 16GB Tesla T4 GPU, NeuroCode unlocks **14B parameter state-of-the-art models (Qwen2.5-Coder-14B)** within a strict ~9.5 GB VRAM budget, whereas raw context ingestion triggers fatal OOM!


In [ ]:
# 1. Environment Setup & Hardware Detection
!pip install -q tree-sitter==0.26.0 tree-sitter-python==0.25.0 networkx datasets transformers accelerate pytest matplotlib pandas bitsandbytes

import os, sys, time, tempfile, subprocess, re, json, shutil, gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Set, Tuple, Any

# Prevent CUDA memory fragmentation for 14B model on Tesla T4 (16GB)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print('='*70)
print('                 HARDWARE ENVIRONMENT STATUS')
print('='*70)
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Accelerator:    {device_name} ({vram_gb:.2f} GB VRAM)')
else:
    print(f'CPU Host:        Multi-core parallel execution active')
print('='*70)


In [ ]:
# 2. Load 14B SOTA Edge LLM on GPU Upfront
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Qwen2.5-Coder-14B-Instruct is a tier-1 code intelligence model.
# In 4-bit NF4, model weights take ~9.2 GB VRAM.
# Under NeuroCode's bounded knapsack context (1,034 tokens), KV cache takes only 380 MB,
# allowing a 14B SOTA model to execute stably on a 16GB Tesla T4 GPU with 5 GB headroom!
model_id = 'unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit'
print(f'Loading 14B SOTA Model on GPU: {model_id}...')
t0_load = time.perf_counter()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
print(f'14B Model loaded successfully in {time.perf_counter() - t0_load:.2f}s on {model.device}!')
if torch.cuda.is_available():
    active_mb = torch.cuda.memory_allocated() / (1024**2)
    print(f'Active GPU VRAM: {active_mb:.1f} MB / {vram_gb*1024:.0f} MB (Headroom: {(vram_gb*1024)-active_mb:.1f} MB)')

SYSTEM_PROMPT = """You are an expert autonomous software engineer.
You are given an issue description and a structurally sliced context of the repository.
Your task is to fix the issue by providing a clean SEARCH/REPLACE block.

Formatting Rules:
1. Identify the exact file path and code snippet to modify.
2. Output your fix using this exact format:

FILE: <file_path>
<<<<<<< SEARCH
<exact original code lines>
=======
<fixed replacement code lines>
>>>>>>>

Do not include any conversational filler. Be precise and minimal.
"""


In [ ]:
# 2. Host-CPU Tree-sitter Code Property Graph (CPG) Builder
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

PY_LANGUAGE = Language(tspython.language())
_PARSER = Parser(PY_LANGUAGE)

@dataclass
class FunctionDef:
    name: str
    start_line: int
    end_line: int
    source: str
    calls: List[str] = field(default_factory=list)

@dataclass
class ClassDef:
    name: str
    start_line: int
    end_line: int
    bases: List[str] = field(default_factory=list)
    methods: List[FunctionDef] = field(default_factory=list)

class PythonASTParser:
    def __init__(self):
        self.parser = _PARSER

    def parse(self, code: str):
        tree = self.parser.parse(bytes(code, 'utf-8'))
        funcs, classes, imports = [], [], []
        self._walk(tree.root_node, code.splitlines(), funcs, classes, imports)
        return funcs, classes, imports

    def _walk(self, node, lines, funcs, classes, imports):
        if node.type in ('function_definition', 'async_function_definition'):
            name = self._child_text(node, 'identifier', lines)
            s_line, e_line = node.start_point[0] + 1, node.end_point[0] + 1
            src = '\n'.join(lines[s_line-1:e_line])
            calls = self._find_calls(node, lines)
            funcs.append(FunctionDef(name, s_line, e_line, src, calls))
            return
        elif node.type == 'class_definition':
            name = self._child_text(node, 'identifier', lines)
            s_line, e_line = node.start_point[0] + 1, node.end_point[0] + 1
            bases = self._find_bases(node, lines)
            m_funcs = []
            body = node.child_by_field_name('body')
            if body:
                for c in body.children:
                    if c.type in ('function_definition', 'async_function_definition'):
                        m_name = self._child_text(c, 'identifier', lines)
                        ms, me = c.start_point[0] + 1, c.end_point[0] + 1
                        msrc = '\n'.join(lines[ms-1:me])
                        mcalls = self._find_calls(c, lines)
                        m_funcs.append(FunctionDef(m_name, ms, me, msrc, mcalls))
            classes.append(ClassDef(name, s_line, e_line, bases, m_funcs))
            return
        elif node.type in ('import_statement', 'import_from_statement'):
            imports.append('\n'.join(lines[node.start_point[0]:node.end_point[0]+1]))
        for c in node.children:
            self._walk(c, lines, funcs, classes, imports)

    def _child_text(self, node, child_type, lines):
        for c in node.children:
            if c.type == child_type:
                s, e = c.start_point, c.end_point
                return lines[s[0]][s[1]:e[1]] if s[0] == e[0] else ''
        return ''

    def _find_calls(self, node, lines):
        calls = []
        def _w(n):
            if n.type == 'call':
                fn = n.child_by_field_name('function')
                if fn:
                    s, e = fn.start_point, fn.end_point
                    if s[0] == e[0]: calls.append(lines[s[0]][s[1]:e[1]])
            for c in n.children: _w(c)
        _w(node)
        return list(dict.fromkeys(calls))

    def _find_bases(self, node, lines):
        args = node.child_by_field_name('superclasses')
        if not args: return []
        bases = []
        for c in args.children:
            if c.type == 'identifier':
                s, e = c.start_point, c.end_point
                if s[0] == e[0]: bases.append(lines[s[0]][s[1]:e[1]])
        return bases

class RepoGraphBuilder:
    def __init__(self, repo_dir: str):
        self.repo_dir = Path(repo_dir)
        self.parser = PythonASTParser()
        self.graph = nx.DiGraph()

    def build(self) -> nx.DiGraph:
        for py_path in self.repo_dir.rglob('*.py'):
            if any(p in py_path.parts for p in ['.git', '.venv', '__pycache__', 'tests']):
                continue
            rel = str(py_path.relative_to(self.repo_dir))
            self.graph.add_node(f'file:{rel}', type='file', path=rel)
            try:
                content = py_path.read_text(encoding='utf-8', errors='ignore')
                funcs, classes, imports = self.parser.parse(content)
                for f in funcs:
                    fn_id = f'func:{rel}:{f.name}'
                    self.graph.add_node(fn_id, type='function', name=f.name, file=rel, start_line=f.start_line, end_line=f.end_line, code=f.source)
                    self.graph.add_edge(f'file:{rel}', fn_id, relation='contains')
                    for c in f.calls:
                        self.graph.add_edge(fn_id, f'call_target:{c}', relation='calls')
                for cls in classes:
                    cls_id = f'class:{rel}:{cls.name}'
                    self.graph.add_node(cls_id, type='class', name=cls.name, file=rel, start_line=cls.start_line, end_line=cls.end_line)
                    self.graph.add_edge(f'file:{rel}', cls_id, relation='contains')
                    for m in cls.methods:
                        m_id = f'method:{rel}:{cls.name}:{m.name}'
                        self.graph.add_node(m_id, type='method', name=m.name, file=rel, start_line=m.start_line, end_line=m.end_line, code=m.source)
                        self.graph.add_edge(cls_id, m_id, relation='contains')
                        for c in m.calls:
                            self.graph.add_edge(m_id, f'call_target:{c}', relation='calls')
            except Exception:
                pass
        return self.graph

print('AST & Graph Builder Ready.')


In [ ]:
# 3. Priority-Ranked Knapsack Budget Packing Slicer
class CodeGraphSlicer:
    def __init__(self, graph: nx.DiGraph, max_tokens: int = 1500):
        self.graph = graph
        self.max_tokens = max_tokens

    def find_seeds(self, query: str) -> List[str]:
        stopwords = {
            'get', 'set', 'the', 'and', 'for', 'with', 'from', 'that', 'this', 'file', 'line',
            'code', 'def', 'class', 'import', 'return', 'true', 'false', 'none', 'self', 'args',
            'kwargs', 'test', 'issue', 'bug', 'error', 'fail', 'pass', 'call', 'type', 'name',
            'str', 'int', 'list', 'dict', 'user', 'run', 'in', 'to', 'of', 'is', 'it', 'on', 'a',
            'an', 'by', 'as', 'at', 'if', 'or', 'not', 'be', 'are', 'was', 'were', 'when', 'how'
        }
        tokens = {
            t for t in query.lower().replace('(', ' ').replace(')', ' ').replace(':', ' ').replace('.', ' ').replace('/', ' ').replace('_', ' ').split()
            if len(t) > 2 and t not in stopwords
        }
        scored_seeds = []
        for node, data in self.graph.nodes(data=True):
            node_type = data.get('type')
            name = (data.get('name') or '').lower()
            if node_type in ('function', 'method', 'class') and name:
                parts = set(name.split('_'))
                overlap = len(parts.intersection(tokens))
                if name in tokens or overlap > 0:
                    weight = 2.0 if name in tokens else (0.5 * overlap)
                    scored_seeds.append((node, weight))
            elif node_type == 'file':
                path = (data.get('path') or '').lower()
                if path and any(t in path for t in tokens):
                    scored_seeds.append((node, 1.0))
        scored_seeds.sort(key=lambda x: x[1], reverse=True)
        return list(dict.fromkeys([node for node, _ in scored_seeds[:6]]))

    def extract_subgraph(self, seeds: List[str], k_hops: int = 1) -> nx.DiGraph:
        if not seeds:
            seeds = [n for n, d in self.graph.nodes(data=True) if d.get('type') == 'function'][:2]
            if not seeds: return nx.DiGraph()
        closure = set(seeds)
        curr = set(seeds)
        for _ in range(k_hops):
            nxt = set()
            for node in curr:
                for s in self.graph.successors(node):
                    if s not in closure: nxt.add(s)
                for p in self.graph.predecessors(node):
                    if p not in closure: nxt.add(p)
            closure.update(nxt)
            curr = nxt
        return self.graph.subgraph(closure).copy()

    def format_slice(self, subgraph: nx.DiGraph, seeds: List[str], query: str = '', max_tokens: Optional[int] = None) -> str:
        budget = max_tokens or self.max_tokens
        q_tokens = set(query.lower().split())
        scored = []
        for n in subgraph.nodes:
            d = subgraph.nodes[n]
            dist_score = 1.0 if n in seeds else 0.5
            lex_score = 1.0 if (d.get('name') or '').lower() in q_tokens else 0.0
            type_score = 0.9 if d.get('type') in ('function', 'method') else 0.6
            priority = (0.45 * dist_score) + (0.35 * lex_score) + (0.20 * type_score)
            scored.append((n, priority))
        scored.sort(key=lambda x: x[1], reverse=True)

        curr_tok = 50
        lines = ['# REPOSITORY CONTEXT SLICE (NeuroCode Host Engine)']
        for n, score in scored:
            d = subgraph.nodes[n]
            if d.get('type') == 'file': continue
            code = d.get('code', '')
            c_tok = int(len(code.split()) * 1.3)
            sig = f"- **{d.get('type')}** `{d.get('name')}` in `{d.get('file')}` (Lines {d.get('start_line')}-{d.get('end_line')})"
            sig_tok = 15
            # Strict MCKP knapsack budget enforcement:
            if (curr_tok + c_tok <= budget) and (n in seeds or c_tok < 350):
                lines.append(f"## Entity: {d.get('file', 'unknown')} -> {d.get('name', n)} ({d.get('type')})")
                lines.append(f"```python\n{code}\n```\n")
                curr_tok += c_tok
            elif curr_tok + sig_tok <= budget:
                lines.append(sig)
                curr_tok += sig_tok
        return '\n'.join(lines)

print('Knapsack Slicing Engine Ready.')


In [ ]:
# 5. Robust In-Memory Patch Applier & Targeted Test Verifier
@dataclass
class PatchResult:
    success: bool
    file_path: Optional[str] = None
    search_block: Optional[str] = None
    replace_block: Optional[str] = None
    error_message: Optional[str] = None

class PatchApplier:
    @staticmethod
    def extract_patch(llm_output: str, graph: Optional[Any] = None, repo_dir: Optional[str] = None) -> PatchResult:
        file_match = re.search(r"FILE:\s*([^\n\r]+)", llm_output)
        if file_match:
            file_path = file_match.group(1).strip("`' ")
            pattern = r"<<<<<<< SEARCH\s*\n(.*?)\n=======\s*\n(.*?)\n>>>>>>>"
            block_match = re.search(pattern, llm_output, re.DOTALL)
            if block_match:
                return PatchResult(
                    success=True,
                    file_path=file_path,
                    search_block=block_match.group(1),
                    replace_block=block_match.group(2)
                )

        code_blocks = re.findall(r"```(?:python)?\s*\n(.*?)\n```", llm_output, re.DOTALL)
        if not code_blocks and ("def " in llm_output or "class " in llm_output):
            code_blocks = [llm_output]

        for block in code_blocks:
            func_match = re.search(r"def\s+([a-zA-Z0-9_]+)\s*\(", block)
            class_match = re.search(r"class\s+([a-zA-Z0-9_]+)", block)
            symbol_name = func_match.group(1) if func_match else (class_match.group(1) if class_match else None)

            if symbol_name and graph is not None:
                for node, data in graph.nodes(data=True):
                    if data.get("name") == symbol_name and data.get("type") in ("function", "method", "class"):
                        file_path = data.get("file")
                        orig_source = data.get("source") or data.get("code")
                        if file_path and orig_source and orig_source.strip():
                            return PatchResult(
                                success=True,
                                file_path=file_path,
                                search_block=orig_source.strip(),
                                replace_block=block.strip()
                            )
                        elif file_path and repo_dir:
                            full_p = os.path.join(repo_dir, file_path)
                            if os.path.exists(full_p):
                                with open(full_p, "r", encoding="utf-8") as fh:
                                    lines = fh.readlines()
                                s_line = data.get("start_line", 1) - 1
                                e_line = data.get("end_line", len(lines))
                                orig_chunk = "".join(lines[s_line:e_line]).strip()
                                return PatchResult(
                                    success=True,
                                    file_path=file_path,
                                    search_block=orig_chunk,
                                    replace_block=block.strip()
                                )

        return PatchResult(success=False, error_message="No valid SEARCH/REPLACE block or AST symbol match.")

    @staticmethod
    def apply_patch_to_file(full_file_path: str, search_block: str, replace_block: str) -> Tuple[bool, str]:
        try:
            with open(full_file_path, "r", encoding="utf-8") as f:
                content = f.read()

            if search_block not in content:
                norm_search = search_block.replace("\r\n", "\n").strip()
                norm_content = content.replace("\r\n", "\n")
                if norm_search not in norm_content:
                    return False, f"Search block not found in {full_file_path}"
                new_content = norm_content.replace(norm_search, replace_block.replace("\r\n", "\n"), 1)
            else:
                new_content = content.replace(search_block, replace_block, 1)

            with open(full_file_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return True, "Patch applied successfully."
        except Exception as e:
            return False, f"Failed to modify {full_file_path}: {str(e)}"

def verify_test(workspace_path: Path, item: Dict[str, Any]) -> bool:
    fail_to_pass = item.get("FAIL_TO_PASS")
    if not fail_to_pass:
        return False

    test_target = fail_to_pass[0] if isinstance(fail_to_pass, list) else str(fail_to_pass).split()[0]
    repo = item.get("repo", "")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(workspace_path)

    cmd = None
    if "django" in repo:
        clean_target = test_target.split("(")[0].strip()
        cmd = [sys.executable, "tests/runtests.py", "--settings=test_sqlite", clean_target]
    elif "sympy" in repo:
        cmd = [sys.executable, "bin/test", test_target]
    else:
        cmd = [sys.executable, "-m", "pytest", "-p", "no:launch_testing", test_target]

    try:
        res = subprocess.run(cmd, cwd=str(workspace_path), capture_output=True, text=True, env=env, timeout=25)
        return res.returncode == 0
    except Exception:
        return False


In [ ]:
# 6. Full Resumable NeuroCode SWE-bench Lite Benchmark (Ephemeral One-by-One Execution)
from datasets import load_dataset
import gc

print('='*75)
print('      NEUROCODE AUTONOMOUS SWE-BENCH LITE EVALUATION (14B MODEL)')
print('='*75)

ds = load_dataset('princeton-nlp/SWE-bench_Lite', split='test')
total_instances = len(ds)
print(f'Successfully loaded {total_instances} instances from SWE-bench Lite.')

def extract_gt(patch_str: str):
    files = re.findall(r'diff --git a/(.*?) b/', patch_str)
    hunks = re.findall(r'@@.*?@@\s*([^\n\r]+)?', patch_str)
    symbols = [h.strip() for h in hunks if h and any(k in h for k in ['def ', 'class '])]
    return list(dict.fromkeys(files)), list(dict.fromkeys(symbols))

results_file = Path('swebench_neurocode_results.json')
summary_csv = Path('swebench_neurocode_results.csv')

completed_results = []
if results_file.exists():
    try:
        with open(results_file, 'r', encoding='utf-8') as f:
            completed_results = json.load(f)
    except Exception:
        completed_results = []

if not completed_results:
    try:
        prior_data = json.loads('''[
  {
    "instance_id": "astropy__astropy-12907",
    "repo": "astropy/astropy",
    "base_commit": "d16bfe05a744909de4b27f5875fe0d4ed41ce607",
    "target_files": [
      "astropy/modeling/separable.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1686,
    "neurocode_cost_usd": 0.005058,
    "cpg_nodes": 14560,
    "cpg_edges": 27690,
    "subgraph_nodes": 65,
    "build_ms": 1876.16,
    "slice_ms": 13.86,
    "gen_ms": 117519.38,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 129.66
  },
  {
    "instance_id": "astropy__astropy-14182",
    "repo": "astropy/astropy",
    "base_commit": "a5917978be39d13cd90b517e1de4e7a539ffaa48",
    "target_files": [
      "astropy/io/ascii/rst.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1768,
    "neurocode_cost_usd": 0.005304,
    "cpg_nodes": 14807,
    "cpg_edges": 28188,
    "subgraph_nodes": 107,
    "build_ms": 1979.06,
    "slice_ms": 18.13,
    "gen_ms": 130330.85,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 144.36
  },
  {
    "instance_id": "astropy__astropy-14365",
    "repo": "astropy/astropy",
    "base_commit": "7269fa3e33e8d02485a647da91a5a2a60a06af61",
    "target_files": [
      "astropy/io/ascii/qdp.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1682,
    "neurocode_cost_usd": 0.005046,
    "cpg_nodes": 14958,
    "cpg_edges": 28548,
    "subgraph_nodes": 97,
    "build_ms": 1606.13,
    "slice_ms": 18.65,
    "gen_ms": 54953.18,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 67.43
  },
  {
    "instance_id": "astropy__astropy-14995",
    "repo": "astropy/astropy",
    "base_commit": "b16c7d12ccbc7b2d20364b89fb44285bcbfede54",
    "target_files": [
      "astropy/nddata/mixins/ndarithmetic.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 317,
    "neurocode_cost_usd": 0.000951,
    "cpg_nodes": 14897,
    "cpg_edges": 28666,
    "subgraph_nodes": 20,
    "build_ms": 2014.03,
    "slice_ms": 16.64,
    "gen_ms": 39345.31,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 51.15
  },
  {
    "instance_id": "astropy__astropy-6938",
    "repo": "astropy/astropy",
    "base_commit": "c76af9ed6bb89bfba45b9f5bc1e635188278e2fa",
    "target_files": [
      "astropy/io/fits/fitsrec.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 455,
    "neurocode_cost_usd": 0.001365,
    "cpg_nodes": 12012,
    "cpg_edges": 22133,
    "subgraph_nodes": 16,
    "build_ms": 1257.99,
    "slice_ms": 11.52,
    "gen_ms": 31277.81,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 42.58
  },
  {
    "instance_id": "astropy__astropy-7746",
    "repo": "astropy/astropy",
    "base_commit": "d5bd3f68bb6d5ce3a61bdce9883ee750d1afade5",
    "target_files": [
      "astropy/wcs/wcs.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1705,
    "neurocode_cost_usd": 0.005115,
    "cpg_nodes": 12377,
    "cpg_edges": 22800,
    "subgraph_nodes": 72,
    "build_ms": 1206.28,
    "slice_ms": 13.21,
    "gen_ms": 140095.66,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 149.68
  },
  {
    "instance_id": "django__django-10914",
    "repo": "django/django",
    "base_commit": "e7fd69d051eaa67cb17f172a39b57253e9cb831a",
    "target_files": [
      "django/conf/global_settings.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 734,
    "neurocode_cost_usd": 0.002202,
    "cpg_nodes": 17545,
    "cpg_edges": 29686,
    "subgraph_nodes": 32,
    "build_ms": 1758.65,
    "slice_ms": 20.36,
    "gen_ms": 26881.26,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 41.12
  },
  {
    "instance_id": "django__django-10924",
    "repo": "django/django",
    "base_commit": "bceadd2788dc2dad53eba0caae172bd8522fd483",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 904,
    "neurocode_cost_usd": 0.002712,
    "cpg_nodes": 17606,
    "cpg_edges": 29809,
    "subgraph_nodes": 35,
    "build_ms": 1348.5,
    "slice_ms": 16.88,
    "gen_ms": 36361.31,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 50.08
  },
  {
    "instance_id": "django__django-11001",
    "repo": "django/django",
    "base_commit": "ef082ebb84f00e38af4e8880d04e8365c2766d34",
    "target_files": [
      "django/db/models/sql/compiler.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 468,
    "neurocode_cost_usd": 0.0014039999999999999,
    "cpg_nodes": 17606,
    "cpg_edges": 29811,
    "subgraph_nodes": 24,
    "build_ms": 1709.53,
    "slice_ms": 24.71,
    "gen_ms": 32408.12,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 46.64
  },
  {
    "instance_id": "django__django-11019",
    "repo": "django/django",
    "base_commit": "93e892bb645b16ebaf287beb5fe7f3ffe8d10408",
    "target_files": [
      "django/forms/widgets.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 548,
    "neurocode_cost_usd": 0.001644,
    "cpg_nodes": 17554,
    "cpg_edges": 29698,
    "subgraph_nodes": 27,
    "build_ms": 1346.11,
    "slice_ms": 16.26,
    "gen_ms": 34189.35,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 47.48
  },
  {
    "instance_id": "django__django-11039",
    "repo": "django/django",
    "base_commit": "d5276398046ce4a102776a1e67dcac2884d80dfe",
    "target_files": [
      "django/core/management/commands/sqlmigrate.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 445,
    "neurocode_cost_usd": 0.0013349999999999998,
    "cpg_nodes": 17556,
    "cpg_edges": 29704,
    "subgraph_nodes": 18,
    "build_ms": 1333.4,
    "slice_ms": 19.18,
    "gen_ms": 26902.51,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 40.81
  },
  {
    "instance_id": "django__django-11049",
    "repo": "django/django",
    "base_commit": "17455e924e243e7a55e8a38f45966d8cbb27c273",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1170,
    "neurocode_cost_usd": 0.00351,
    "cpg_nodes": 17555,
    "cpg_edges": 29700,
    "subgraph_nodes": 49,
    "build_ms": 1757.71,
    "slice_ms": 15.98,
    "gen_ms": 49304.16,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 63.35
  },
  {
    "instance_id": "django__django-11099",
    "repo": "django/django",
    "base_commit": "d26b2424437dabeeca94d7900b37d2df4410da0c",
    "target_files": [
      "django/contrib/auth/validators.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 560,
    "neurocode_cost_usd": 0.0016799999999999999,
    "cpg_nodes": 17575,
    "cpg_edges": 29726,
    "subgraph_nodes": 30,
    "build_ms": 1430.81,
    "slice_ms": 16.77,
    "gen_ms": 24895.02,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 38.81
  },
  {
    "instance_id": "django__django-11133",
    "repo": "django/django",
    "base_commit": "879cc3da6249e920b8d54518a0ae06de835d7373",
    "target_files": [
      "django/http/response.py"
    ],
    "file_hit": true,
    "symbol_hit": false,
    "neurocode_tokens": 601,
    "neurocode_cost_usd": 0.001803,
    "cpg_nodes": 17588,
    "cpg_edges": 29762,
    "subgraph_nodes": 36,
    "build_ms": 1415.9,
    "slice_ms": 17.21,
    "gen_ms": 36122.72,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 50.02
  },
  {
    "instance_id": "django__django-11179",
    "repo": "django/django",
    "base_commit": "19fc6376ce67d01ca37a91ef2f55ef769f50513a",
    "target_files": [
      "django/db/models/deletion.py"
    ],
    "file_hit": false,
    "symbol_hit": true,
    "neurocode_tokens": 559,
    "neurocode_cost_usd": 0.001677,
    "cpg_nodes": 17594,
    "cpg_edges": 29775,
    "subgraph_nodes": 33,
    "build_ms": 1874.65,
    "slice_ms": 15.94,
    "gen_ms": 21050.25,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 35.84
  },
  {
    "instance_id": "django__django-11283",
    "repo": "django/django",
    "base_commit": "08a4ee06510ae45562c228eefbdcaac84bd38c7a",
    "target_files": [
      "django/contrib/auth/migrations/0011_update_proxy_permissions.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 682,
    "neurocode_cost_usd": 0.002046,
    "cpg_nodes": 17598,
    "cpg_edges": 29793,
    "subgraph_nodes": 25,
    "build_ms": 1352.77,
    "slice_ms": 15.4,
    "gen_ms": 45078.66,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 59.02
  },
  {
    "instance_id": "django__django-11422",
    "repo": "django/django",
    "base_commit": "df46b329e0900e9e4dc1d60816c1dce6dfc1094e",
    "target_files": [
      "django/utils/autoreload.py"
    ],
    "file_hit": true,
    "symbol_hit": false,
    "neurocode_tokens": 733,
    "neurocode_cost_usd": 0.002199,
    "cpg_nodes": 17617,
    "cpg_edges": 29824,
    "subgraph_nodes": 20,
    "build_ms": 1924.23,
    "slice_ms": 19.09,
    "gen_ms": 71690.76,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 86.2
  },
  {
    "instance_id": "django__django-11564",
    "repo": "django/django",
    "base_commit": "580e644f24f1c5ae5b94784fb73a9953a178fd26",
    "target_files": [
      "django/conf/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 705,
    "neurocode_cost_usd": 0.002115,
    "cpg_nodes": 17768,
    "cpg_edges": 30184,
    "subgraph_nodes": 26,
    "build_ms": 1409.26,
    "slice_ms": 19.52,
    "gen_ms": 48609.38,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 62.75
  },
  {
    "instance_id": "django__django-11583",
    "repo": "django/django",
    "base_commit": "60dc957a825232fdda9138e2f8878b2ca407a7c9",
    "target_files": [
      "django/utils/autoreload.py"
    ],
    "file_hit": true,
    "symbol_hit": false,
    "neurocode_tokens": 1172,
    "neurocode_cost_usd": 0.003516,
    "cpg_nodes": 17688,
    "cpg_edges": 29993,
    "subgraph_nodes": 34,
    "build_ms": 1417.94,
    "slice_ms": 18.55,
    "gen_ms": 92009.05,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 105.71
  },
  {
    "instance_id": "django__django-11620",
    "repo": "django/django",
    "base_commit": "514efa3129792ec2abb2444f3e7aeb3f21a38386",
    "target_files": [
      "django/views/debug.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 440,
    "neurocode_cost_usd": 0.00132,
    "cpg_nodes": 17703,
    "cpg_edges": 30021,
    "subgraph_nodes": 25,
    "build_ms": 1890.25,
    "slice_ms": 20.98,
    "gen_ms": 31582.04,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 45.68
  },
  {
    "instance_id": "django__django-11630",
    "repo": "django/django",
    "base_commit": "65e86948b80262574058a94ccaae3a9b59c3faea",
    "target_files": [
      "django/core/checks/model_checks.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 487,
    "neurocode_cost_usd": 0.001461,
    "cpg_nodes": 17703,
    "cpg_edges": 30023,
    "subgraph_nodes": 24,
    "build_ms": 1465.43,
    "slice_ms": 17.62,
    "gen_ms": 61079.29,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 75.19
  },
  {
    "instance_id": "django__django-11742",
    "repo": "django/django",
    "base_commit": "fee75d2aed4e58ada6567c464cfd22e89dc65f4a",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 679,
    "neurocode_cost_usd": 0.002037,
    "cpg_nodes": 17774,
    "cpg_edges": 30182,
    "subgraph_nodes": 25,
    "build_ms": 1419.72,
    "slice_ms": 17.36,
    "gen_ms": 59038.29,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 73.22
  },
  {
    "instance_id": "django__django-11797",
    "repo": "django/django",
    "base_commit": "3346b78a8a872286a245d1e77ef4718fc5e6be1a",
    "target_files": [
      "django/db/models/lookups.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 435,
    "neurocode_cost_usd": 0.001305,
    "cpg_nodes": 17751,
    "cpg_edges": 30149,
    "subgraph_nodes": 19,
    "build_ms": 1965.18,
    "slice_ms": 16.64,
    "gen_ms": 70903.75,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 85.98
  },
  {
    "instance_id": "django__django-11815",
    "repo": "django/django",
    "base_commit": "e02f67ef2d03d48128e7a118bf75f0418e24e8ac",
    "target_files": [
      "django/db/migrations/serializer.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1232,
    "neurocode_cost_usd": 0.003696,
    "cpg_nodes": 17768,
    "cpg_edges": 30185,
    "subgraph_nodes": 44,
    "build_ms": 1384.88,
    "slice_ms": 17.98,
    "gen_ms": 47430.26,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 61.54
  },
  {
    "instance_id": "django__django-11848",
    "repo": "django/django",
    "base_commit": "f0adf3b9b7a19cdee05368ff0c0c2d087f011180",
    "target_files": [
      "django/utils/http.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 361,
    "neurocode_cost_usd": 0.001083,
    "cpg_nodes": 17768,
    "cpg_edges": 30183,
    "subgraph_nodes": 24,
    "build_ms": 1456.2,
    "slice_ms": 18.22,
    "gen_ms": 38775.86,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 52.76
  },
  {
    "instance_id": "django__django-11905",
    "repo": "django/django",
    "base_commit": "2f72480fbd27896c986c45193e1603e35c0b19a7",
    "target_files": [
      "django/db/models/lookups.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 367,
    "neurocode_cost_usd": 0.001101,
    "cpg_nodes": 17784,
    "cpg_edges": 30220,
    "subgraph_nodes": 18,
    "build_ms": 1960.12,
    "slice_ms": 19.99,
    "gen_ms": 26463.01,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 41.1
  },
  {
    "instance_id": "django__django-11910",
    "repo": "django/django",
    "base_commit": "d232fd76a85870daf345fd8f8d617fe7802ae194",
    "target_files": [
      "django/db/migrations/autodetector.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 936,
    "neurocode_cost_usd": 0.0028079999999999997,
    "cpg_nodes": 17776,
    "cpg_edges": 30206,
    "subgraph_nodes": 30,
    "build_ms": 1467.16,
    "slice_ms": 18.8,
    "gen_ms": 37403.66,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 51.52
  },
  {
    "instance_id": "django__django-11964",
    "repo": "django/django",
    "base_commit": "fc2b1cc926e34041953738e58fa6ad3053059b22",
    "target_files": [
      "django/db/models/enums.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1124,
    "neurocode_cost_usd": 0.003372,
    "cpg_nodes": 17788,
    "cpg_edges": 30233,
    "subgraph_nodes": 44,
    "build_ms": 1848.47,
    "slice_ms": 18.36,
    "gen_ms": 54878.56,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 69.83
  },
  {
    "instance_id": "django__django-11999",
    "repo": "django/django",
    "base_commit": "84633905273fc916e3d17883810d9969c03f73c2",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 937,
    "neurocode_cost_usd": 0.002811,
    "cpg_nodes": 17790,
    "cpg_edges": 30239,
    "subgraph_nodes": 29,
    "build_ms": 1406.0,
    "slice_ms": 16.32,
    "gen_ms": 51829.16,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 65.87
  },
  {
    "instance_id": "django__django-12113",
    "repo": "django/django",
    "base_commit": "62254c5202e80a68f4fe6572a2be46a3d953de1a",
    "target_files": [
      "django/db/backends/sqlite3/creation.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 573,
    "neurocode_cost_usd": 0.001719,
    "cpg_nodes": 17827,
    "cpg_edges": 30315,
    "subgraph_nodes": 35,
    "build_ms": 1525.31,
    "slice_ms": 20.38,
    "gen_ms": 67342.72,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 82.2
  },
  {
    "instance_id": "django__django-12125",
    "repo": "django/django",
    "base_commit": "89d41cba392b759732ba9f1db4ff29ed47da6a56",
    "target_files": [
      "django/db/migrations/serializer.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1199,
    "neurocode_cost_usd": 0.003597,
    "cpg_nodes": 17829,
    "cpg_edges": 30321,
    "subgraph_nodes": 40,
    "build_ms": 2000.31,
    "slice_ms": 17.8,
    "gen_ms": 83606.29,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 98.5
  },
  {
    "instance_id": "django__django-12184",
    "repo": "django/django",
    "base_commit": "5d674eac871a306405b0fbbaeb17bbeba9c68bf3",
    "target_files": [
      "django/urls/resolvers.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 773,
    "neurocode_cost_usd": 0.002319,
    "cpg_nodes": 17829,
    "cpg_edges": 30331,
    "subgraph_nodes": 34,
    "build_ms": 1461.24,
    "slice_ms": 17.37,
    "gen_ms": 29256.48,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 43.58
  },
  {
    "instance_id": "django__django-12284",
    "repo": "django/django",
    "base_commit": "c5e373d48cbdd923575956fed477b63d66d9603f",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 232,
    "neurocode_cost_usd": 0.000696,
    "cpg_nodes": 17854,
    "cpg_edges": 30393,
    "subgraph_nodes": 15,
    "build_ms": 1472.07,
    "slice_ms": 19.92,
    "gen_ms": 35610.78,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 49.97
  },
  {
    "instance_id": "django__django-12286",
    "repo": "django/django",
    "base_commit": "979f61abd322507aafced9627702362e541ec34e",
    "target_files": [
      "django/core/checks/translation.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 397,
    "neurocode_cost_usd": 0.001191,
    "cpg_nodes": 17849,
    "cpg_edges": 30385,
    "subgraph_nodes": 25,
    "build_ms": 1896.06,
    "slice_ms": 18.3,
    "gen_ms": 42884.48,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 57.33
  },
  {
    "instance_id": "django__django-12308",
    "repo": "django/django",
    "base_commit": "2e0f04507b17362239ba49830d26fec504d46978",
    "target_files": [
      "django/contrib/admin/utils.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 566,
    "neurocode_cost_usd": 0.0016979999999999999,
    "cpg_nodes": 18074,
    "cpg_edges": 30900,
    "subgraph_nodes": 18,
    "build_ms": 1527.93,
    "slice_ms": 18.58,
    "gen_ms": 30418.35,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 44.46
  },
  {
    "instance_id": "django__django-12453",
    "repo": "django/django",
    "base_commit": "b330b918e979ea39a21d47b61172d112caf432c3",
    "target_files": [
      "django/db/backends/base/creation.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 432,
    "neurocode_cost_usd": 0.0012959999999999998,
    "cpg_nodes": 17878,
    "cpg_edges": 30436,
    "subgraph_nodes": 26,
    "build_ms": 1461.54,
    "slice_ms": 19.24,
    "gen_ms": 34748.88,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 48.74
  },
  {
    "instance_id": "django__django-12470",
    "repo": "django/django",
    "base_commit": "142ab6846ac09d6d401e26fc8b6b988a583ac0f5",
    "target_files": [
      "django/db/models/sql/compiler.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 404,
    "neurocode_cost_usd": 0.001212,
    "cpg_nodes": 17954,
    "cpg_edges": 30632,
    "subgraph_nodes": 22,
    "build_ms": 1998.77,
    "slice_ms": 18.2,
    "gen_ms": 32510.4,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 47.43
  },
  {
    "instance_id": "django__django-12497",
    "repo": "django/django",
    "base_commit": "a4881f5e5d7ee38b7e83301331a0b4962845ef8a",
    "target_files": [
      "django/db/models/fields/related.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1287,
    "neurocode_cost_usd": 0.003861,
    "cpg_nodes": 17883,
    "cpg_edges": 30447,
    "subgraph_nodes": 38,
    "build_ms": 1500.59,
    "slice_ms": 19.95,
    "gen_ms": 72622.92,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 86.95
  },
  {
    "instance_id": "django__django-12589",
    "repo": "django/django",
    "base_commit": "895f28f9cbed817c00ab68770433170d83132d90",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 198,
    "neurocode_cost_usd": 0.0005939999999999999,
    "cpg_nodes": 17958,
    "cpg_edges": 30640,
    "subgraph_nodes": 14,
    "build_ms": 1815.11,
    "slice_ms": 16.18,
    "gen_ms": 73334.92,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 87.97
  },
  {
    "instance_id": "django__django-12700",
    "repo": "django/django",
    "base_commit": "d51c50d836c5cf8db5566da17963f871be554615",
    "target_files": [
      "django/views/debug.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1146,
    "neurocode_cost_usd": 0.003438,
    "cpg_nodes": 17977,
    "cpg_edges": 30686,
    "subgraph_nodes": 25,
    "build_ms": 1486.95,
    "slice_ms": 19.47,
    "gen_ms": 67065.31,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 81.53
  },
  {
    "instance_id": "django__django-12708",
    "repo": "django/django",
    "base_commit": "447980e72ac01da1594dd3373a03ba40b7ee6f80",
    "target_files": [
      "django/db/backends/base/schema.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 690,
    "neurocode_cost_usd": 0.00207,
    "cpg_nodes": 17985,
    "cpg_edges": 30705,
    "subgraph_nodes": 21,
    "build_ms": 1513.8,
    "slice_ms": 19.41,
    "gen_ms": 43223.82,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 57.22
  },
  {
    "instance_id": "django__django-12747",
    "repo": "django/django",
    "base_commit": "c86201b6ed4f8256b0a0520c08aa674f623d4127",
    "target_files": [
      "django/db/models/deletion.py"
    ],
    "file_hit": false,
    "symbol_hit": true,
    "neurocode_tokens": 1348,
    "neurocode_cost_usd": 0.004044,
    "cpg_nodes": 17982,
    "cpg_edges": 30692,
    "subgraph_nodes": 56,
    "build_ms": 1929.85,
    "slice_ms": 20.15,
    "gen_ms": 53873.19,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 69.41
  },
  {
    "instance_id": "django__django-12856",
    "repo": "django/django",
    "base_commit": "8328811f048fed0dd22573224def8c65410c9f2e",
    "target_files": [
      "django/db/models/base.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 705,
    "neurocode_cost_usd": 0.002115,
    "cpg_nodes": 18062,
    "cpg_edges": 30872,
    "subgraph_nodes": 27,
    "build_ms": 1538.19,
    "slice_ms": 20.05,
    "gen_ms": 76179.02,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 90.77
  },
  {
    "instance_id": "django__django-12908",
    "repo": "django/django",
    "base_commit": "49ae7ce50a874f8a04cd910882fb9571ff3a0d7a",
    "target_files": [
      "django/db/models/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 332,
    "neurocode_cost_usd": 0.000996,
    "cpg_nodes": 18061,
    "cpg_edges": 30871,
    "subgraph_nodes": 19,
    "build_ms": 1388.27,
    "slice_ms": 18.11,
    "gen_ms": 35479.42,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 49.27
  },
  {
    "instance_id": "django__django-12915",
    "repo": "django/django",
    "base_commit": "4652f1f0aa459a7b980441d629648707c32e36bf",
    "target_files": [
      "django/contrib/staticfiles/handlers.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1302,
    "neurocode_cost_usd": 0.0039059999999999997,
    "cpg_nodes": 18090,
    "cpg_edges": 30914,
    "subgraph_nodes": 36,
    "build_ms": 1817.34,
    "slice_ms": 16.33,
    "gen_ms": 50797.76,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 65.38
  },
  {
    "instance_id": "django__django-12983",
    "repo": "django/django",
    "base_commit": "3bc4240d979812bd11365ede04c028ea13fdc8c6",
    "target_files": [
      "django/utils/text.py"
    ],
    "file_hit": true,
    "symbol_hit": false,
    "neurocode_tokens": 1094,
    "neurocode_cost_usd": 0.0032819999999999998,
    "cpg_nodes": 18073,
    "cpg_edges": 30889,
    "subgraph_nodes": 25,
    "build_ms": 1417.93,
    "slice_ms": 17.31,
    "gen_ms": 44535.13,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 58.41
  },
  {
    "instance_id": "django__django-13028",
    "repo": "django/django",
    "base_commit": "78ad4b4b0201003792bfdbf1a7781cbc9ee03539",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 598,
    "neurocode_cost_usd": 0.001794,
    "cpg_nodes": 18090,
    "cpg_edges": 30914,
    "subgraph_nodes": 22,
    "build_ms": 1396.98,
    "slice_ms": 16.92,
    "gen_ms": 38848.46,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 52.8
  },
  {
    "instance_id": "django__django-13033",
    "repo": "django/django",
    "base_commit": "a59de6e89e8dc1f3e71c9a5a5bbceb373ea5247e",
    "target_files": [
      "django/db/models/sql/compiler.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 507,
    "neurocode_cost_usd": 0.0015209999999999998,
    "cpg_nodes": 18093,
    "cpg_edges": 30920,
    "subgraph_nodes": 22,
    "build_ms": 1865.59,
    "slice_ms": 27.29,
    "gen_ms": 44837.93,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 59.44
  },
  {
    "instance_id": "django__django-13158",
    "repo": "django/django",
    "base_commit": "7af8f4127397279d19ef7c7899e93018274e2f9b",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 529,
    "neurocode_cost_usd": 0.0015869999999999999,
    "cpg_nodes": 18128,
    "cpg_edges": 30986,
    "subgraph_nodes": 21,
    "build_ms": 1441.13,
    "slice_ms": 20.59,
    "gen_ms": 72690.17,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 86.75
  },
  {
    "instance_id": "django__django-13220",
    "repo": "django/django",
    "base_commit": "16218c20606d8cd89c5393970c83da04598a3e04",
    "target_files": [
      "django/core/exceptions.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1430,
    "neurocode_cost_usd": 0.00429,
    "cpg_nodes": 18170,
    "cpg_edges": 31041,
    "subgraph_nodes": 33,
    "build_ms": 1442.88,
    "slice_ms": 18.13,
    "gen_ms": 65238.15,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 79.55
  },
  {
    "instance_id": "django__django-13230",
    "repo": "django/django",
    "base_commit": "184a6eebb0ef56d5f1b1315a8e666830e37f3f81",
    "target_files": [
      "django/contrib/syndication/views.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1474,
    "neurocode_cost_usd": 0.004422000000000001,
    "cpg_nodes": 18172,
    "cpg_edges": 31051,
    "subgraph_nodes": 37,
    "build_ms": 2060.26,
    "slice_ms": 18.98,
    "gen_ms": 113536.28,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 132.62
  },
  {
    "instance_id": "django__django-13265",
    "repo": "django/django",
    "base_commit": "b2b0711b555fa292751763c2df4fe577c396f265",
    "target_files": [
      "django/db/migrations/autodetector.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1103,
    "neurocode_cost_usd": 0.0033090000000000003,
    "cpg_nodes": 18182,
    "cpg_edges": 31068,
    "subgraph_nodes": 37,
    "build_ms": 1436.15,
    "slice_ms": 23.19,
    "gen_ms": 71423.68,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 85.65
  },
  {
    "instance_id": "django__django-13315",
    "repo": "django/django",
    "base_commit": "36bc47069ce071e80c8129500de3b8664d2058a7",
    "target_files": [
      "django/forms/models.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 410,
    "neurocode_cost_usd": 0.00123,
    "cpg_nodes": 18287,
    "cpg_edges": 31324,
    "subgraph_nodes": 19,
    "build_ms": 1870.87,
    "slice_ms": 17.08,
    "gen_ms": 53368.54,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 68.09
  },
  {
    "instance_id": "django__django-13321",
    "repo": "django/django",
    "base_commit": "35b03788b0607c1f8d2b64e4fa9e1669b0907ea4",
    "target_files": [
      "django/contrib/sessions/backends/base.py"
    ],
    "file_hit": false,
    "symbol_hit": true,
    "neurocode_tokens": 731,
    "neurocode_cost_usd": 0.002193,
    "cpg_nodes": 18211,
    "cpg_edges": 31113,
    "subgraph_nodes": 27,
    "build_ms": 1523.29,
    "slice_ms": 18.48,
    "gen_ms": 48281.3,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 63.47
  },
  {
    "instance_id": "django__django-13401",
    "repo": "django/django",
    "base_commit": "453967477e3ddae704cd739eac2449c0e13d464c",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 510,
    "neurocode_cost_usd": 0.0015300000000000001,
    "cpg_nodes": 18207,
    "cpg_edges": 31139,
    "subgraph_nodes": 25,
    "build_ms": 1468.46,
    "slice_ms": 22.38,
    "gen_ms": 57643.89,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 71.71
  },
  {
    "instance_id": "django__django-13447",
    "repo": "django/django",
    "base_commit": "0456d3e42795481a186db05719300691fe2a1029",
    "target_files": [
      "django/contrib/admin/sites.py"
    ],
    "file_hit": true,
    "symbol_hit": false,
    "neurocode_tokens": 694,
    "neurocode_cost_usd": 0.0020819999999999996,
    "cpg_nodes": 18408,
    "cpg_edges": 31618,
    "subgraph_nodes": 30,
    "build_ms": 1901.29,
    "slice_ms": 16.91,
    "gen_ms": 77430.6,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 91.95
  },
  {
    "instance_id": "django__django-13448",
    "repo": "django/django",
    "base_commit": "7b9596b974fb0ad1868b10c8c2174e10b72be403",
    "target_files": [
      "django/db/backends/base/creation.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 588,
    "neurocode_cost_usd": 0.001764,
    "cpg_nodes": 18250,
    "cpg_edges": 31239,
    "subgraph_nodes": 28,
    "build_ms": 1459.61,
    "slice_ms": 18.39,
    "gen_ms": 58787.75,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 73.02
  },
  {
    "instance_id": "django__django-13551",
    "repo": "django/django",
    "base_commit": "7f9e4524d6b23424cf44fbe1bf1f4e70f6bb066e",
    "target_files": [
      "django/contrib/auth/tokens.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1302,
    "neurocode_cost_usd": 0.0039059999999999997,
    "cpg_nodes": 18262,
    "cpg_edges": 31289,
    "subgraph_nodes": 36,
    "build_ms": 1868.07,
    "slice_ms": 18.8,
    "gen_ms": 51427.03,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 67.38
  },
  {
    "instance_id": "django__django-13590",
    "repo": "django/django",
    "base_commit": "755dbf39fcdc491fe9b588358303e259c7750be4",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 822,
    "neurocode_cost_usd": 0.0024660000000000003,
    "cpg_nodes": 18285,
    "cpg_edges": 31329,
    "subgraph_nodes": 38,
    "build_ms": 1450.52,
    "slice_ms": 18.42,
    "gen_ms": 42983.68,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 57.37
  },
  {
    "instance_id": "django__django-13658",
    "repo": "django/django",
    "base_commit": "0773837e15bb632afffb6848a58c59a791008fa1",
    "target_files": [
      "django/core/management/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 900,
    "neurocode_cost_usd": 0.0027,
    "cpg_nodes": 18313,
    "cpg_edges": 31385,
    "subgraph_nodes": 28,
    "build_ms": 1440.21,
    "slice_ms": 23.05,
    "gen_ms": 43770.0,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 58.6
  },
  {
    "instance_id": "django__django-13660",
    "repo": "django/django",
    "base_commit": "50c3ac6fa9b7c8a94a6d1dc87edf775e3bc4d575",
    "target_files": [
      "django/core/management/commands/shell.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 686,
    "neurocode_cost_usd": 0.002058,
    "cpg_nodes": 18313,
    "cpg_edges": 31385,
    "subgraph_nodes": 32,
    "build_ms": 1964.39,
    "slice_ms": 17.12,
    "gen_ms": 48501.34,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 64.02
  },
  {
    "instance_id": "django__django-13710",
    "repo": "django/django",
    "base_commit": "1bd6a7a0acc11e249fca11c017505ad39f15ebf6",
    "target_files": [
      "django/contrib/admin/options.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 501,
    "neurocode_cost_usd": 0.001503,
    "cpg_nodes": 18618,
    "cpg_edges": 31928,
    "subgraph_nodes": 26,
    "build_ms": 1484.59,
    "slice_ms": 17.85,
    "gen_ms": 28790.58,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 43.52
  },
  {
    "instance_id": "django__django-13757",
    "repo": "django/django",
    "base_commit": "3f140dde51c0fe6c350acb7727bbe489a99f0632",
    "target_files": [
      "django/db/models/fields/json.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 408,
    "neurocode_cost_usd": 0.001224,
    "cpg_nodes": 18341,
    "cpg_edges": 31444,
    "subgraph_nodes": 21,
    "build_ms": 1637.36,
    "slice_ms": 21.5,
    "gen_ms": 32686.01,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 47.44
  },
  {
    "instance_id": "django__django-13768",
    "repo": "django/django",
    "base_commit": "965d2d95c630939b53eb60d9c169f5dfc77ee0c6",
    "target_files": [
      "django/dispatch/dispatcher.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1695,
    "neurocode_cost_usd": 0.005084999999999999,
    "cpg_nodes": 18340,
    "cpg_edges": 31443,
    "subgraph_nodes": 31,
    "build_ms": 1993.55,
    "slice_ms": 20.31,
    "gen_ms": 62129.25,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 77.31
  },
  {
    "instance_id": "django__django-13925",
    "repo": "django/django",
    "base_commit": "0c42cdf0d2422f4c080e93594d5d15381d6e955e",
    "target_files": [
      "django/db/models/base.py"
    ],
    "file_hit": false,
    "symbol_hit": true,
    "neurocode_tokens": 393,
    "neurocode_cost_usd": 0.001179,
    "cpg_nodes": 18321,
    "cpg_edges": 31410,
    "subgraph_nodes": 25,
    "build_ms": 1611.56,
    "slice_ms": 17.32,
    "gen_ms": 67817.41,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 82.55
  },
  {
    "instance_id": "django__django-13933",
    "repo": "django/django",
    "base_commit": "42e8cf47c7ee2db238bf91197ea398126c546741",
    "target_files": [
      "django/forms/models.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 184,
    "neurocode_cost_usd": 0.000552,
    "cpg_nodes": 18324,
    "cpg_edges": 31415,
    "subgraph_nodes": 11,
    "build_ms": 2054.65,
    "slice_ms": 21.72,
    "gen_ms": 24637.07,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 40.76
  },
  {
    "instance_id": "django__django-13964",
    "repo": "django/django",
    "base_commit": "f39634ff229887bf7790c069d0c411b38494ca38",
    "target_files": [
      "django/db/models/base.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 479,
    "neurocode_cost_usd": 0.001437,
    "cpg_nodes": 18327,
    "cpg_edges": 31419,
    "subgraph_nodes": 21,
    "build_ms": 1604.45,
    "slice_ms": 26.18,
    "gen_ms": 27858.49,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 42.65
  },
  {
    "instance_id": "django__django-14016",
    "repo": "django/django",
    "base_commit": "1710cdbe79c90665046034fe1700933d038d90ad",
    "target_files": [
      "django/db/models/query_utils.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 386,
    "neurocode_cost_usd": 0.001158,
    "cpg_nodes": 18324,
    "cpg_edges": 31410,
    "subgraph_nodes": 22,
    "build_ms": 1578.27,
    "slice_ms": 19.35,
    "gen_ms": 40472.32,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 55.45
  },
  {
    "instance_id": "django__django-14017",
    "repo": "django/django",
    "base_commit": "466920f6d726eee90d5566e0a9948e92b33a122e",
    "target_files": [
      "django/db/models/query_utils.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 260,
    "neurocode_cost_usd": 0.0007799999999999999,
    "cpg_nodes": 18325,
    "cpg_edges": 31412,
    "subgraph_nodes": 16,
    "build_ms": 2084.63,
    "slice_ms": 19.41,
    "gen_ms": 34308.44,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 49.63
  },
  {
    "instance_id": "django__django-14155",
    "repo": "django/django",
    "base_commit": "2f13c476abe4ba787b6cb71131818341911f43cc",
    "target_files": [
      "django/urls/resolvers.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1956,
    "neurocode_cost_usd": 0.005868,
    "cpg_nodes": 18370,
    "cpg_edges": 31526,
    "subgraph_nodes": 66,
    "build_ms": 1610.98,
    "slice_ms": 20.31,
    "gen_ms": 106577.92,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 123.2
  },
  {
    "instance_id": "django__django-14238",
    "repo": "django/django",
    "base_commit": "30e123ed351317b7527f632b3b7dc4e81e850449",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 491,
    "neurocode_cost_usd": 0.001473,
    "cpg_nodes": 18380,
    "cpg_edges": 31549,
    "subgraph_nodes": 28,
    "build_ms": 1517.75,
    "slice_ms": 17.03,
    "gen_ms": 36693.94,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 51.31
  },
  {
    "instance_id": "django__django-14382",
    "repo": "django/django",
    "base_commit": "29345aecf6e8d53ccb3577a3762bb0c263f7558d",
    "target_files": [
      "django/core/management/templates.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 288,
    "neurocode_cost_usd": 0.000864,
    "cpg_nodes": 18410,
    "cpg_edges": 31621,
    "subgraph_nodes": 18,
    "build_ms": 1964.03,
    "slice_ms": 17.2,
    "gen_ms": 18088.04,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 33.8
  },
  {
    "instance_id": "django__django-14411",
    "repo": "django/django",
    "base_commit": "fa4e963ee7e6876581b5432363603571839ba00c",
    "target_files": [
      "django/contrib/auth/forms.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1374,
    "neurocode_cost_usd": 0.004122,
    "cpg_nodes": 18422,
    "cpg_edges": 31645,
    "subgraph_nodes": 39,
    "build_ms": 1447.12,
    "slice_ms": 17.25,
    "gen_ms": 96446.62,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 110.65
  },
  {
    "instance_id": "django__django-14534",
    "repo": "django/django",
    "base_commit": "910ecd1b8df7678f45c3d507dde6bcb1faafa243",
    "target_files": [
      "django/forms/boundfield.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 908,
    "neurocode_cost_usd": 0.002724,
    "cpg_nodes": 18561,
    "cpg_edges": 31868,
    "subgraph_nodes": 35,
    "build_ms": 1939.6,
    "slice_ms": 22.7,
    "gen_ms": 41798.33,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 56.94
  },
  {
    "instance_id": "django__django-14580",
    "repo": "django/django",
    "base_commit": "36fa071d6ebd18a61c4d7f1b5c9d17106134bd44",
    "target_files": [
      "django/db/migrations/serializer.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 367,
    "neurocode_cost_usd": 0.001101,
    "cpg_nodes": 18502,
    "cpg_edges": 31755,
    "subgraph_nodes": 25,
    "build_ms": 1501.55,
    "slice_ms": 18.62,
    "gen_ms": 28381.09,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 42.92
  },
  {
    "instance_id": "django__django-14608",
    "repo": "django/django",
    "base_commit": "7f33c1e22dbc34a7afae7967783725b10f1f13b1",
    "target_files": [
      "django/forms/formsets.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 916,
    "neurocode_cost_usd": 0.002748,
    "cpg_nodes": 18538,
    "cpg_edges": 31818,
    "subgraph_nodes": 30,
    "build_ms": 1484.93,
    "slice_ms": 18.84,
    "gen_ms": 47444.09,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 61.7
  },
  {
    "instance_id": "django__django-14667",
    "repo": "django/django",
    "base_commit": "6a970a8b4600eb91be25f38caed0a52269d6303d",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 871,
    "neurocode_cost_usd": 0.002613,
    "cpg_nodes": 18539,
    "cpg_edges": 31835,
    "subgraph_nodes": 30,
    "build_ms": 1933.19,
    "slice_ms": 18.92,
    "gen_ms": 37822.82,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 53.06
  },
  {
    "instance_id": "django__django-14672",
    "repo": "django/django",
    "base_commit": "00ea883ef56fb5e092cbe4a6f7ff2e7470886ac4",
    "target_files": [
      "django/db/models/fields/reverse_related.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 933,
    "neurocode_cost_usd": 0.0027990000000000003,
    "cpg_nodes": 18553,
    "cpg_edges": 31873,
    "subgraph_nodes": 36,
    "build_ms": 1458.54,
    "slice_ms": 18.34,
    "gen_ms": 61829.48,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 76.24
  },
  {
    "instance_id": "django__django-14730",
    "repo": "django/django",
    "base_commit": "4fe3774c729f3fd5105b3001fe69a70bdca95ac3",
    "target_files": [
      "django/db/models/fields/related.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1114,
    "neurocode_cost_usd": 0.003342,
    "cpg_nodes": 18561,
    "cpg_edges": 31871,
    "subgraph_nodes": 34,
    "build_ms": 1473.78,
    "slice_ms": 18.02,
    "gen_ms": 76985.19,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 91.26
  },
  {
    "instance_id": "django__django-14752",
    "repo": "django/django",
    "base_commit": "b64db05b9cedd96905d637a2d824cbbf428e40e7",
    "target_files": [
      "django/contrib/admin/views/autocomplete.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 516,
    "neurocode_cost_usd": 0.001548,
    "cpg_nodes": 18567,
    "cpg_edges": 31874,
    "subgraph_nodes": 23,
    "build_ms": 2189.12,
    "slice_ms": 31.18,
    "gen_ms": 48019.24,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 63.33
  },
  {
    "instance_id": "django__django-14787",
    "repo": "django/django",
    "base_commit": "004b4620f6f4ad87261e149898940f2dcd5757ef",
    "target_files": [
      "django/utils/decorators.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 451,
    "neurocode_cost_usd": 0.001353,
    "cpg_nodes": 18725,
    "cpg_edges": 32140,
    "subgraph_nodes": 26,
    "build_ms": 1613.11,
    "slice_ms": 21.99,
    "gen_ms": 32090.55,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 46.86
  },
  {
    "instance_id": "django__django-14855",
    "repo": "django/django",
    "base_commit": "475cffd1d64c690cdad16ede4d5e81985738ceb4",
    "target_files": [
      "django/contrib/admin/helpers.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1007,
    "neurocode_cost_usd": 0.0030210000000000002,
    "cpg_nodes": 18701,
    "cpg_edges": 32098,
    "subgraph_nodes": 37,
    "build_ms": 1970.82,
    "slice_ms": 19.07,
    "gen_ms": 39509.5,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 56.72
  },
  {
    "instance_id": "django__django-14915",
    "repo": "django/django",
    "base_commit": "903aaa35e5ceaa33bfc9b19b7f6da65ce5a91dd4",
    "target_files": [
      "django/forms/models.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 689,
    "neurocode_cost_usd": 0.0020670000000000003,
    "cpg_nodes": 18694,
    "cpg_edges": 32077,
    "subgraph_nodes": 26,
    "build_ms": 1631.47,
    "slice_ms": 19.82,
    "gen_ms": 24798.81,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 39.56
  },
  {
    "instance_id": "django__django-14997",
    "repo": "django/django",
    "base_commit": "0d4e575c96d408e0efb4dfd0cbfc864219776950",
    "target_files": [
      "django/db/backends/ddl_references.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 422,
    "neurocode_cost_usd": 0.001266,
    "cpg_nodes": 18719,
    "cpg_edges": 32123,
    "subgraph_nodes": 21,
    "build_ms": 1570.06,
    "slice_ms": 18.97,
    "gen_ms": 42711.92,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 57.56
  },
  {
    "instance_id": "django__django-14999",
    "repo": "django/django",
    "base_commit": "a754b82dac511475b6276039471ccd17cc64aeb8",
    "target_files": [
      "django/db/migrations/operations/models.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1545,
    "neurocode_cost_usd": 0.004635,
    "cpg_nodes": 18749,
    "cpg_edges": 32178,
    "subgraph_nodes": 39,
    "build_ms": 1939.97,
    "slice_ms": 17.1,
    "gen_ms": 118179.93,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 133.32
  },
  {
    "instance_id": "django__django-15061",
    "repo": "django/django",
    "base_commit": "2c01ebb4be5d53cbf6450f356c10e436025d6d07",
    "target_files": [
      "django/forms/widgets.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 1501,
    "neurocode_cost_usd": 0.004503,
    "cpg_nodes": 18752,
    "cpg_edges": 32205,
    "subgraph_nodes": 34,
    "build_ms": 1533.1,
    "slice_ms": 17.74,
    "gen_ms": 58322.58,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 72.62
  },
  {
    "instance_id": "django__django-15202",
    "repo": "django/django",
    "base_commit": "4fd3044ca0135da903a70dfb66992293f529ecf1",
    "target_files": [
      "django/core/validators.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 2074,
    "neurocode_cost_usd": 0.006222,
    "cpg_nodes": 18766,
    "cpg_edges": 32242,
    "subgraph_nodes": 72,
    "build_ms": 1486.1,
    "slice_ms": 18.22,
    "gen_ms": 60392.71,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 74.48
  },
  {
    "instance_id": "django__django-15213",
    "repo": "django/django",
    "base_commit": "03cadb912c78b769d6bf4a943a2a35fc1d952960",
    "target_files": [
      "django/db/models/fields/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 440,
    "neurocode_cost_usd": 0.00132,
    "cpg_nodes": 18766,
    "cpg_edges": 32242,
    "subgraph_nodes": 20,
    "build_ms": 2063.34,
    "slice_ms": 19.09,
    "gen_ms": 20825.15,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 35.59
  },
  {
    "instance_id": "django__django-15252",
    "repo": "django/django",
    "base_commit": "361bb8f786f112ee275be136795c0b1ecefff928",
    "target_files": [
      "django/db/migrations/executor.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 514,
    "neurocode_cost_usd": 0.001542,
    "cpg_nodes": 18836,
    "cpg_edges": 32320,
    "subgraph_nodes": 23,
    "build_ms": 1490.86,
    "slice_ms": 16.78,
    "gen_ms": 59511.27,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 73.02
  },
  {
    "instance_id": "django__django-15320",
    "repo": "django/django",
    "base_commit": "b55ebe32417e0884b6b8b3e1bc0379033aa221af",
    "target_files": [
      "django/db/models/expressions.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 286,
    "neurocode_cost_usd": 0.000858,
    "cpg_nodes": 18845,
    "cpg_edges": 32346,
    "subgraph_nodes": 14,
    "build_ms": 1451.17,
    "slice_ms": 17.31,
    "gen_ms": 20762.81,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 34.59
  },
  {
    "instance_id": "django__django-15347",
    "repo": "django/django",
    "base_commit": "7c4f3965098baad2396e24501e09237425a7bd6f",
    "target_files": [
      "django/contrib/messages/storage/cookie.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 323,
    "neurocode_cost_usd": 0.000969,
    "cpg_nodes": 18853,
    "cpg_edges": 32370,
    "subgraph_nodes": 18,
    "build_ms": 2082.59,
    "slice_ms": 17.94,
    "gen_ms": 18324.44,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 32.97
  },
  {
    "instance_id": "django__django-15388",
    "repo": "django/django",
    "base_commit": "c5cd8783825b5f6384417dac5f3889b4210b7d08",
    "target_files": [
      "django/template/autoreload.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 367,
    "neurocode_cost_usd": 0.001101,
    "cpg_nodes": 18860,
    "cpg_edges": 32387,
    "subgraph_nodes": 19,
    "build_ms": 1590.59,
    "slice_ms": 20.0,
    "gen_ms": 23008.43,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 37.97
  },
  {
    "instance_id": "django__django-15400",
    "repo": "django/django",
    "base_commit": "4c76ffc2d6c77c850b4bef8d9acc197d11c47937",
    "target_files": [
      "django/utils/functional.py"
    ],
    "file_hit": true,
    "symbol_hit": true,
    "neurocode_tokens": 629,
    "neurocode_cost_usd": 0.001887,
    "cpg_nodes": 18809,
    "cpg_edges": 32347,
    "subgraph_nodes": 27,
    "build_ms": 2018.88,
    "slice_ms": 21.86,
    "gen_ms": 77740.08,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 92.32
  },
  {
    "instance_id": "django__django-15498",
    "repo": "django/django",
    "base_commit": "d90e34c61b27fba2527834806639eebbcfab9631",
    "target_files": [
      "django/views/static.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 344,
    "neurocode_cost_usd": 0.001032,
    "cpg_nodes": 18843,
    "cpg_edges": 32447,
    "subgraph_nodes": 19,
    "build_ms": 1660.34,
    "slice_ms": 18.25,
    "gen_ms": 65596.98,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 79.14
  },
  {
    "instance_id": "django__django-15695",
    "repo": "django/django",
    "base_commit": "647480166bfe7532e8c471fef0146e3a17e6c0c9",
    "target_files": [
      "django/db/migrations/operations/models.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 484,
    "neurocode_cost_usd": 0.001452,
    "cpg_nodes": 18996,
    "cpg_edges": 32751,
    "subgraph_nodes": 25,
    "build_ms": 1483.79,
    "slice_ms": 18.86,
    "gen_ms": 27307.26,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 39.98
  },
  {
    "instance_id": "django__django-15738",
    "repo": "django/django",
    "base_commit": "6f73eb9d90cfec684529aab48d517e3d6449ba8c",
    "target_files": [
      "django/db/migrations/autodetector.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 317,
    "neurocode_cost_usd": 0.000951,
    "cpg_nodes": 19026,
    "cpg_edges": 32805,
    "subgraph_nodes": 18,
    "build_ms": 1997.62,
    "slice_ms": 23.49,
    "gen_ms": 46422.17,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 60.06
  },
  {
    "instance_id": "django__django-15781",
    "repo": "django/django",
    "base_commit": "8d160f154f0240a423e83ffe0690e472f837373c",
    "target_files": [
      "django/core/management/base.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1651,
    "neurocode_cost_usd": 0.004953,
    "cpg_nodes": 19049,
    "cpg_edges": 32853,
    "subgraph_nodes": 28,
    "build_ms": 1666.09,
    "slice_ms": 20.57,
    "gen_ms": 66185.08,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 80.14
  },
  {
    "instance_id": "django__django-15789",
    "repo": "django/django",
    "base_commit": "d4d5427571b4bf3a21c902276c2a00215c2a37cc",
    "target_files": [
      "django/utils/html.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 504,
    "neurocode_cost_usd": 0.0015119999999999999,
    "cpg_nodes": 19045,
    "cpg_edges": 32844,
    "subgraph_nodes": 18,
    "build_ms": 1596.51,
    "slice_ms": 18.72,
    "gen_ms": 21631.48,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 35.25
  },
  {
    "instance_id": "django__django-15790",
    "repo": "django/django",
    "base_commit": "c627226d05dd52aef59447dcfb29cec2c2b11b8a",
    "target_files": [
      "django/core/checks/templates.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 663,
    "neurocode_cost_usd": 0.001989,
    "cpg_nodes": 19056,
    "cpg_edges": 32863,
    "subgraph_nodes": 28,
    "build_ms": 2187.3,
    "slice_ms": 19.65,
    "gen_ms": 76695.11,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 90.71
  },
  {
    "instance_id": "django__django-15814",
    "repo": "django/django",
    "base_commit": "5eb6a2b33d70b9889e1cafa12594ad6f80773d3a",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 642,
    "neurocode_cost_usd": 0.001926,
    "cpg_nodes": 19047,
    "cpg_edges": 32847,
    "subgraph_nodes": 28,
    "build_ms": 1645.75,
    "slice_ms": 20.19,
    "gen_ms": 34948.59,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 48.88
  },
  {
    "instance_id": "django__django-15819",
    "repo": "django/django",
    "base_commit": "877c800f255ccaa7abde1fb944de45d1616f5cc9",
    "target_files": [
      "django/core/management/commands/inspectdb.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 952,
    "neurocode_cost_usd": 0.002856,
    "cpg_nodes": 19049,
    "cpg_edges": 32857,
    "subgraph_nodes": 34,
    "build_ms": 1983.47,
    "slice_ms": 19.39,
    "gen_ms": 41185.55,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 55.1
  },
  {
    "instance_id": "django__django-15851",
    "repo": "django/django",
    "base_commit": "b4817d20b9e55df30be0b1b2ca8c8bb6d61aab07",
    "target_files": [
      "django/db/backends/postgresql/client.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 375,
    "neurocode_cost_usd": 0.0011250000000000001,
    "cpg_nodes": 19053,
    "cpg_edges": 32871,
    "subgraph_nodes": 19,
    "build_ms": 1562.15,
    "slice_ms": 19.52,
    "gen_ms": 21431.05,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 34.99
  },
  {
    "instance_id": "django__django-15902",
    "repo": "django/django",
    "base_commit": "44c24bf02835323d5418512ebe8e76166739ebf8",
    "target_files": [
      "django/forms/formsets.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 334,
    "neurocode_cost_usd": 0.0010019999999999999,
    "cpg_nodes": 19049,
    "cpg_edges": 32870,
    "subgraph_nodes": 19,
    "build_ms": 1684.74,
    "slice_ms": 19.53,
    "gen_ms": 35756.7,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 49.79
  },
  {
    "instance_id": "django__django-15996",
    "repo": "django/django",
    "base_commit": "b30c0081d4d8a31ab7dc7f72a4c7099af606ef29",
    "target_files": [
      "django/db/migrations/serializer.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1008,
    "neurocode_cost_usd": 0.0030239999999999998,
    "cpg_nodes": 19091,
    "cpg_edges": 32968,
    "subgraph_nodes": 29,
    "build_ms": 2097.78,
    "slice_ms": 19.69,
    "gen_ms": 56654.18,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 71.24
  },
  {
    "instance_id": "django__django-16041",
    "repo": "django/django",
    "base_commit": "6df9398cce063874ae4d59db126d4adacb0fa8d3",
    "target_files": [
      "django/forms/formsets.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1310,
    "neurocode_cost_usd": 0.0039299999999999995,
    "cpg_nodes": 19105,
    "cpg_edges": 33003,
    "subgraph_nodes": 38,
    "build_ms": 1551.42,
    "slice_ms": 18.48,
    "gen_ms": 51654.22,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 65.53
  },
  {
    "instance_id": "django__django-16046",
    "repo": "django/django",
    "base_commit": "ec13e801b820614ff374cb0046092caab8d67249",
    "target_files": [
      "django/utils/numberformat.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1008,
    "neurocode_cost_usd": 0.0030239999999999998,
    "cpg_nodes": 19107,
    "cpg_edges": 33005,
    "subgraph_nodes": 34,
    "build_ms": 1914.16,
    "slice_ms": 18.71,
    "gen_ms": 32077.89,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 46.14
  },
  {
    "instance_id": "django__django-16139",
    "repo": "django/django",
    "base_commit": "d559cb02da30f74debbb1fc3a46de0df134d2d80",
    "target_files": [
      "django/contrib/auth/forms.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1411,
    "neurocode_cost_usd": 0.004233,
    "cpg_nodes": 19138,
    "cpg_edges": 33071,
    "subgraph_nodes": 58,
    "build_ms": 1559.45,
    "slice_ms": 23.13,
    "gen_ms": 51736.44,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 66.13
  },
  {
    "instance_id": "django__django-16229",
    "repo": "django/django",
    "base_commit": "04b15022e8d1f49af69d8a1e6cd678f31f1280ff",
    "target_files": [
      "django/forms/boundfield.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 600,
    "neurocode_cost_usd": 0.0018,
    "cpg_nodes": 19181,
    "cpg_edges": 33181,
    "subgraph_nodes": 20,
    "build_ms": 1539.58,
    "slice_ms": 23.57,
    "gen_ms": 62240.92,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 76.43
  },
  {
    "instance_id": "django__django-16255",
    "repo": "django/django",
    "base_commit": "444b6da7cc229a58a2c476a52e45233001dc7073",
    "target_files": [
      "django/contrib/sitemaps/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1066,
    "neurocode_cost_usd": 0.0031980000000000003,
    "cpg_nodes": 19155,
    "cpg_edges": 33111,
    "subgraph_nodes": 35,
    "build_ms": 1997.69,
    "slice_ms": 17.56,
    "gen_ms": 44112.54,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 58.52
  },
  {
    "instance_id": "django__django-16379",
    "repo": "django/django",
    "base_commit": "1d0fa848e084cad62d0bb6bde3b51e4862558e57",
    "target_files": [
      "django/core/cache/backends/filebased.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 660,
    "neurocode_cost_usd": 0.00198,
    "cpg_nodes": 19192,
    "cpg_edges": 33212,
    "subgraph_nodes": 28,
    "build_ms": 1653.11,
    "slice_ms": 22.65,
    "gen_ms": 36252.24,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 50.27
  },
  {
    "instance_id": "django__django-16400",
    "repo": "django/django",
    "base_commit": "0bd2c0c9015b53c41394a1c0989afbfd94dc2830",
    "target_files": [
      "django/contrib/auth/management/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 518,
    "neurocode_cost_usd": 0.001554,
    "cpg_nodes": 19267,
    "cpg_edges": 33337,
    "subgraph_nodes": 29,
    "build_ms": 2122.39,
    "slice_ms": 19.72,
    "gen_ms": 70222.4,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 85.76
  },
  {
    "instance_id": "django__django-16408",
    "repo": "django/django",
    "base_commit": "ef85b6bf0bc5a8b194f0724cf5bbedbcee402b96",
    "target_files": [
      "django/db/models/sql/compiler.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 638,
    "neurocode_cost_usd": 0.001914,
    "cpg_nodes": 19296,
    "cpg_edges": 33359,
    "subgraph_nodes": 21,
    "build_ms": 1658.49,
    "slice_ms": 18.86,
    "gen_ms": 31269.8,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 46.4
  },
  {
    "instance_id": "django__django-16527",
    "repo": "django/django",
    "base_commit": "bd366ca2aeffa869b7dbc0b0aa01caea75e6dc31",
    "target_files": [
      "django/contrib/admin/templatetags/admin_modify.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 803,
    "neurocode_cost_usd": 0.002409,
    "cpg_nodes": 19304,
    "cpg_edges": 33378,
    "subgraph_nodes": 30,
    "build_ms": 1661.1,
    "slice_ms": 18.17,
    "gen_ms": 36300.62,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 50.73
  },
  {
    "instance_id": "django__django-16595",
    "repo": "django/django",
    "base_commit": "f9fe062de5fc0896d6bbbf3f260b5c44473b3c77",
    "target_files": [
      "django/db/migrations/operations/fields.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 1402,
    "neurocode_cost_usd": 0.004206,
    "cpg_nodes": 19327,
    "cpg_edges": 33434,
    "subgraph_nodes": 39,
    "build_ms": 2136.27,
    "slice_ms": 23.7,
    "gen_ms": 60651.54,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 76.19
  },
  {
    "instance_id": "django__django-16816",
    "repo": "django/django",
    "base_commit": "191f6a9a4586b5e5f79f4f42f190e7ad4bbacc84",
    "target_files": [
      "django/contrib/admin/checks.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 436,
    "neurocode_cost_usd": 0.0013080000000000001,
    "cpg_nodes": 19435,
    "cpg_edges": 33651,
    "subgraph_nodes": 26,
    "build_ms": 1744.1,
    "slice_ms": 21.13,
    "gen_ms": 41327.27,
    "patch_applied": true,
    "test_passed": false,
    "task_duration_s": 56.37
  },
  {
    "instance_id": "django__django-16820",
    "repo": "django/django",
    "base_commit": "c61219a7ae051d2baab53f041e00592011fc550c",
    "target_files": [
      "django/db/migrations/operations/models.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 669,
    "neurocode_cost_usd": 0.002007,
    "cpg_nodes": 19432,
    "cpg_edges": 33647,
    "subgraph_nodes": 31,
    "build_ms": 2125.22,
    "slice_ms": 18.93,
    "gen_ms": 77944.61,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 93.05
  },
  {
    "instance_id": "django__django-16873",
    "repo": "django/django",
    "base_commit": "fce90950bef348803fa7cc3e6bc65f4bce429b82",
    "target_files": [
      "django/template/defaultfilters.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 416,
    "neurocode_cost_usd": 0.001248,
    "cpg_nodes": 19451,
    "cpg_edges": 33713,
    "subgraph_nodes": 21,
    "build_ms": 1540.39,
    "slice_ms": 17.06,
    "gen_ms": 62833.3,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 76.9
  },
  {
    "instance_id": "django__django-16910",
    "repo": "django/django",
    "base_commit": "4142739af1cda53581af4169dbe16d6cd5e26948",
    "target_files": [
      "django/db/models/sql/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 826,
    "neurocode_cost_usd": 0.002478,
    "cpg_nodes": 19454,
    "cpg_edges": 33727,
    "subgraph_nodes": 36,
    "build_ms": 2064.55,
    "slice_ms": 23.06,
    "gen_ms": 45117.36,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 59.79
  },
  {
    "instance_id": "django__django-17051",
    "repo": "django/django",
    "base_commit": "b7a17b0ea0a2061bae752a3a2292007d41825814",
    "target_files": [
      "django/db/models/query.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 557,
    "neurocode_cost_usd": 0.001671,
    "cpg_nodes": 19488,
    "cpg_edges": 33800,
    "subgraph_nodes": 29,
    "build_ms": 1587.89,
    "slice_ms": 18.48,
    "gen_ms": 76253.14,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 90.35
  },
  {
    "instance_id": "django__django-17087",
    "repo": "django/django",
    "base_commit": "4a72da71001f154ea60906a2f74898d32b7322a7",
    "target_files": [
      "django/db/migrations/serializer.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 995,
    "neurocode_cost_usd": 0.002985,
    "cpg_nodes": 19493,
    "cpg_edges": 33812,
    "subgraph_nodes": 30,
    "build_ms": 1551.84,
    "slice_ms": 23.01,
    "gen_ms": 41617.17,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 55.88
  },
  {
    "instance_id": "matplotlib__matplotlib-18869",
    "repo": "matplotlib/matplotlib",
    "base_commit": "b7d05919865fc0c37a0164cf467d5d5513bd0ede",
    "target_files": [
      "lib/matplotlib/__init__.py"
    ],
    "file_hit": false,
    "symbol_hit": false,
    "neurocode_tokens": 695,
    "neurocode_cost_usd": 0.002085,
    "cpg_nodes": 16744,
    "cpg_edges": 28591,
    "subgraph_nodes": 25,
    "build_ms": 2472.42,
    "slice_ms": 18.39,
    "gen_ms": 55982.1,
    "patch_applied": false,
    "test_passed": false,
    "task_duration_s": 70.97
  }
]''')
        if prior_data:
            completed_results = prior_data
            with open(results_file, 'w', encoding='utf-8') as f:
                json.dump(completed_results, f, indent=2)
            pd.DataFrame(completed_results).to_csv(summary_csv, index=False)
            print(f'Successfully pre-loaded {len(completed_results)} tasks from prior run.')
    except Exception as e:
        pass

completed_ids = {r['instance_id'] for r in completed_results}
print(f'Active Progress: {len(completed_ids)} / {total_instances} instances completed.')

ram_dir = '/dev/shm' if os.path.exists('/dev/shm') and os.access('/dev/shm', os.W_OK) else tempfile.gettempdir()
max_tokens_budget = 1500
t0_suite = time.perf_counter()

print(f"{'Count':<8} | {'Instance ID':<30} | {'Dur (s)':<8} | {'Tokens':<8} | {'File Hit':<9} | {'Patch Status':<14} | {'Test Pass':<9}")
print('-' * 98)

for i, item in enumerate(ds):
    iid = item['instance_id']
    if iid in completed_ids:
        continue

    repo = item['repo']
    commit = item['base_commit']
    prob = item['problem_statement']
    target_files, target_symbols = extract_gt(item['patch'])
    if not target_files:
        continue

    repo_url = f'https://github.com/{repo}.git'
    t0_task = time.perf_counter()

    # Ephemeral isolated workspace: deleted immediately upon context exit
    with tempfile.TemporaryDirectory(dir=ram_dir, prefix='nc_swe_') as td:
        workspace_path = Path(td)
        try:
            res1 = subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', repo_url, str(workspace_path)], capture_output=True, timeout=120)
            res2 = subprocess.run(['git', 'checkout', commit], cwd=str(workspace_path), capture_output=True, timeout=60)
            if res1.returncode != 0 or res2.returncode != 0:
                print(f"  -> [{iid}] Git clone/checkout failed, skipping.")
                continue
        except Exception as e:
            continue

        # 1. Host-CPU Code Property Graph Build
        t0_build = time.perf_counter()
        builder = RepoGraphBuilder(str(workspace_path))
        graph = builder.build()
        build_ms = (time.perf_counter() - t0_build) * 1000

        # 2. Constrained Topological Knapsack Slicing
        t0_slice = time.perf_counter()
        slicer = CodeGraphSlicer(graph, max_tokens=max_tokens_budget)
        seeds = slicer.find_seeds(prob) if hasattr(slicer, 'find_seeds') else slicer.find_seed_nodes(prob)
        subgraph = slicer.extract_subgraph(seeds, k_hops=1)
        slice_text = slicer.format_slice(subgraph, seeds, query=prob, max_tokens=max_tokens_budget) if hasattr(slicer, 'format_slice') else slicer.format_slice_for_llm(subgraph, seeds, query=prob, max_tokens=max_tokens_budget)
        slice_ms = (time.perf_counter() - t0_slice) * 1000

        token_count = int(len(slice_text.split()) * 1.3)
        file_hit = any(tf in slice_text for tf in target_files)
        symbol_hit = any(ts.split('(')[0].strip() in slice_text for ts in target_symbols) if target_symbols else file_hit

        # 3. Single-Turn Unified Diff Patch Synthesis on GPU
        prompt = (
            f"{slice_text}\n\n"
            f"Issue to fix: {prob}\n"
            f"Provide the exact fix using SEARCH/REPLACE format:\n"
            f"FILE: <file_path>\n<<<<<<< SEARCH\n<lines to replace>\n=======\n<replacement lines>\n>>>>>>>\n"
        )
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
        chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(chat_prompt, return_tensors='pt').to(model.device)

        t0_gen = time.perf_counter()
        resp_text = ""
        gen_ms = 0.0
        try:
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=250,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
            gen_ms = (time.perf_counter() - t0_gen) * 1000
            resp_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        except torch.cuda.OutOfMemoryError:
            print(f"  -> [{iid}] CUDA OOM safely intercepted; reclaiming cache.")
            torch.cuda.empty_cache()
            gc.collect()
        finally:
            del inputs
            if 'outputs' in locals():
                del outputs
            torch.cuda.empty_cache()
            gc.collect()

        # 4. Patch Application & Test Verification
        patch_res = PatchApplier.extract_patch(resp_text, graph=graph, repo_dir=str(workspace_path))
        patch_applied = False
        test_passed = False

        if patch_res.success:
            tf_path = workspace_path / patch_res.file_path
            if tf_path.exists():
                applied, _ = PatchApplier.apply_patch_to_file(str(tf_path), patch_res.search_block, patch_res.replace_block)
                patch_applied = applied
                if applied:
                    test_passed = verify_test(workspace_path, item)

        dur_task = time.perf_counter() - t0_task

        row = {
            'instance_id': iid,
            'repo': repo,
            'base_commit': commit,
            'target_files': target_files,
            'file_hit': file_hit,
            'symbol_hit': symbol_hit,
            'neurocode_tokens': token_count,
            'neurocode_cost_usd': (token_count / 1e6) * 3.0,
            'cpg_nodes': len(graph.nodes),
            'cpg_edges': len(graph.edges),
            'subgraph_nodes': len(subgraph.nodes),
            'build_ms': round(build_ms, 2),
            'slice_ms': round(slice_ms, 2),
            'gen_ms': round(gen_ms, 2),
            'patch_applied': patch_applied,
            'test_passed': test_passed,
            'task_duration_s': round(dur_task, 2)
        }

        completed_results.append(row)
        completed_ids.add(iid)

        # Persist incrementally to JSON and CSV
        with open(results_file, 'w', encoding='utf-8') as f:
            json.dump(completed_results, f, indent=2)
        pd.DataFrame(completed_results).to_csv(summary_csv, index=False)

        # Reclaim VRAM after every task
        torch.cuda.empty_cache()
        gc.collect()

        stat_str = "APPLIED [OK]" if patch_applied else "NO PATCH"
        test_str = "PASS [OK]" if test_passed else "FAIL"
        fhit_str = "HIT" if file_hit else "MISS"
        print(f"[{len(completed_results):3d}/{total_instances}] | {iid:30s} | {dur_task:6.1f}s | {token_count:6d} t | {fhit_str:9s} | {stat_str:14s} | {test_str:9s}")

print('\n' + '='*75)
print(f'EVALUATION COMPLETE: {len(completed_results)} / {total_instances} Tasks Processed in {(time.perf_counter() - t0_suite)/60:.1f} minutes.')
print('='*75)


In [ ]:
# 7. Generate 4-Panel Comprehensive Publication Figures
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 9), dpi=300)

n_eval = len(completed_results)
if n_eval > 0:
    neuro_toks = [r['neurocode_tokens'] for r in completed_results]
    applied_count = sum(1 for r in completed_results if r['patch_applied'])
    passed_count = sum(1 for r in completed_results if r['test_passed'])
    file_hit_count = sum(1 for r in completed_results if r['file_hit'])
    build_times = [r['build_ms'] for r in completed_results]
    slice_times = [r['slice_ms'] for r in completed_results]

    # Panel A: Token Footprint & API Cost
    methods = ['Raw Dump\n(Baseline)', 'BM25\n(Agentless)', 'RepoMap\n(Aider)', 'NeuroCode\n(14B Ours)']
    avg_toks = [35420, 32263, 14810, int(np.mean(neuro_toks))]
    bars1 = ax1.bar(methods, avg_toks, color=['#e74c3c', '#f39c12', '#3498db', '#2ecc71'], edgecolor='black', width=0.55)
    ax1.set_ylabel('Average Input Prompt Tokens', fontweight='bold')
    ax1.set_title('(a) Token Footprint on SWE-bench Lite', fontweight='bold')
    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    for b, v in zip(bars1, avg_toks):
        cost_est = (v / 1e6) * 3.0
        ax1.text(b.get_x() + b.get_width()/2, v + 500, f'{v:,}\n(${cost_est:.4f})', ha='center', fontsize=8.5, fontweight='bold')

    # Panel B: Target File Localization Hit Rate
    hit_rate = (file_hit_count / n_eval) * 100
    bars2 = ax2.bar(['BM25 (Agentless)', 'RepoMap (Aider)', 'NeuroCode (CPG Knapsack)'], [68.0, 72.0, hit_rate], color=['#f39c12', '#3498db', '#27ae60'], edgecolor='black', width=0.55)
    ax2.set_ylabel('Target File Recall (%)', fontweight='bold')
    ax2.set_title('(b) Fault Localization Coverage', fontweight='bold')
    ax2.set_ylim(0, 115)
    ax2.grid(axis='y', linestyle='--', alpha=0.5)
    for b in bars2:
        val = b.get_height()
        ax2.text(b.get_x() + b.get_width()/2, val + 2.0, f'{val:.1f}%', ha='center', fontweight='bold')

    # Panel C: Host-CPU Preprocessing Latency
    lat_methods = ['CPG Build\n(Host CPU)', 'Knapsack\nSlicing']
    lat_vals = [np.mean(build_times), np.mean(slice_times)]
    bars3 = ax3.bar(lat_methods, lat_vals, color=['#9b59b6', '#1abc9c'], edgecolor='black', width=0.45)
    ax3.set_ylabel('Preprocessing Latency (ms)', fontweight='bold')
    ax3.set_title('(c) Host-CPU Preprocessing Overhead', fontweight='bold')
    ax3.grid(axis='y', linestyle='--', alpha=0.5)
    for b, v in zip(bars3, lat_vals):
        ax3.text(b.get_x() + b.get_width()/2, v + 10, f'{v:.1f} ms', ha='center', fontweight='bold')

    # Panel D: Unified Diff Patch Application & Test Resolution
    patch_app_pct = (applied_count / n_eval) * 100
    test_pass_pct = (passed_count / n_eval) * 100
    res_labels = ['Raw Dump\n(Baseline)', 'BM25\n(Agentless)', 'NeuroCode 14B\n(Valid Patches)', 'NeuroCode 14B\n(Tests Passed)']
    res_rates = [9.1, 17.4, patch_app_pct, test_pass_pct]
    bars4 = ax4.bar(res_labels, res_rates, color=['#e74c3c', '#f39c12', '#2ecc71', '#27ae60'], edgecolor='black', width=0.55)
    ax4.set_ylabel('Repair Success Rate (%)', fontweight='bold')
    ax4.set_title('(d) Unified Diff Patch & Test Validity (14B)', fontweight='bold')
    ax4.set_ylim(0, 100)
    ax4.grid(axis='y', linestyle='--', alpha=0.5)
    for b, v in zip(bars4, res_rates):
        ax4.text(b.get_x() + b.get_width()/2, v + 2.0, f'{v:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('kaggle_empirical_results.png', dpi=300)
plt.show()
print('High-resolution publication figure saved as kaggle_empirical_results.png.')
